# الفصل 45: التقييم النوعي وتحليل الأخطاء

> الأرقام الإجمالية (85% على MMLU) تُخفي الكثير.

- **الجزء:** 9 — التقييم الشامل
- **الفصل:** 45
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 44: اختبار السلامة والأمان: Red Teaming
- **الوسوم:** Error Analysis، Slice-based Evaluation، Emergent Abilities

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** matplotlib
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("matplotlib", "matplotlib")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-09-evaluation%2Fch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-09-evaluation/ch45-qualitative-evaluation.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> الأرقام الإجمالية (85% على MMLU) تُخفي الكثير.

> **بيانات الفصل**
>
> - **الجزء:** 9 — التقييم الشامل
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 44: اختبار السلامة والأمان: Red Teaming


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم تحليل الأخطاء المنهجي (Error Analysis Taxonomy)
- فهم تحليل الأداء حسب الشرائح (Slice-based Evaluation)
- فهم Error Analysis


## 1. لماذا يهمّك هذا الموضوع؟

الأرقام الإجمالية (85% على MMLU) تُخفي الكثير. هذا الفصل يعلّمك **تشريح** أداء النموذج: أين يفشل بالضبط، لماذا، وكيف تكتشف القدرات غير المتوقعة (الناشئة) أو الضعف الممنهج.


## 2. الخريطة الذهنية


```text
التقييم النوعي = تشريح الفشل
│
├── Error Taxonomy
│   ├── واقعي / استدلالي / تعليمات / تنسيق
│   ├── هلوسة / رفض غير مبرر / لغة / سياق
│   └── severity 1-5
│
├── Slice-based Evaluation
│   ├── حسب اللغة/اللهجة
│   ├── حسب طول السياق
│   ├── حسب نوع المهمة
│   └── حسب مستوى الصعوبة
│
├── Emergent Abilities
│   ├── قفزات عند حجم معين
│   └── قد تكون artifact من مقياس حاد
│
└── أدوات
    ├── تصنيف يدوي (20-50 حالة/أسبوع)
    ├── LLM-assisted annotation
    └── لوحات شرائح
```


## 3. الشرح المتدرج

### 🟢 أ) تحليل الأخطاء المنهجي (Error Analysis Taxonomy)


In [ ]:
from enum import Enum
from dataclasses import dataclass

class ErrorType(Enum):
    FACTUAL_ERROR = "خطأ واقعي"
    REASONING_ERROR = "خطأ منطقي/استدلالي"
    INSTRUCTION_FOLLOWING = "عدم اتّباع التعليمات"
    FORMATTING_ERROR = "خطأ في التنسيق المطلوب"
    HALLUCINATION = "هلوسة (اختلاق معلومة)"
    REFUSAL_UNWARRANTED = "رفض غير مبرر"
    LANGUAGE_QUALITY = "ضعف في جودة اللغة"
    CONTEXT_MISUNDERSTANDING = "سوء فهم السياق"

@dataclass
class ErrorAnnotation:
    prompt: str
    response: str
    error_types: list
    severity: int  # 1 (طفيف) - 5 (حرج)
    notes: str

def build_error_taxonomy_report(annotated_failures: list) -> dict:
    """تجميع إحصائي لأنماط الفشل — يكشف أين يجب التركيز في التحسين"""
    from collections import Counter
    error_counts = Counter()
    for failure in annotated_failures:
        for error_type in failure.error_types:
            error_counts[error_type.value] += 1

    total = len(annotated_failures)
    report = {
        error_type: {"count": count, "percentage": count / total * 100}
        for error_type, count in error_counts.most_common()
    }
    return report


> 💡 **نصيحة عملية**: خصّص لكل مشروع LLM **جلسة أسبوعية لمراجعة الأخطاء يدويًا** (Error Analysis Session) على 20-50 حالة فشل حقيقية من الإنتاج أو التقييم. هذا التمرين البسيط يكشف أنماطًا منهجية (مثل: "النموذج يفشل باستمرار في أسئلة تتطلب حساب التواريخ") لا تظهر أبدًا في الأرقام الإجمالية.

### 🟡 ب) تحليل الأداء حسب الشرائح (Slice-based Evaluation)


In [ ]:
def slice_based_evaluation(model, test_set: list, slice_functions: dict) -> dict:
    """
    تقييم الأداء ليس فقط إجمالًا، بل عبر "شرائح" محدَّدة من البيانات
    (مثال: حسب طول السؤال، اللغة، الموضوع، صعوبة السؤال)
    """
    results_by_slice = {}

    for slice_name, slice_fn in slice_functions.items():
        slice_examples = [ex for ex in test_set if slice_fn(ex)]
        if not slice_examples:
            continue
        correct = sum(1 for ex in slice_examples if model.predict(ex["input"]) == ex["label"])
        results_by_slice[slice_name] = {
            "accuracy": correct / len(slice_examples),
            "n_samples": len(slice_examples)
        }

    return results_by_slice

# مثال على تعريف شرائح مفيدة للغة العربية تحديدًا
arabic_slices = {
    "فصحى": lambda ex: ex.get("dialect") == "msa",
    "لهجة_خليجية": lambda ex: ex.get("dialect") == "gulf",
    "لهجة_مصرية": lambda ex: ex.get("dialect") == "egyptian",
    "أسئلة_قصيرة": lambda ex: len(ex["input"].split()) < 10,
    "أسئلة_طويلة": lambda ex: len(ex["input"].split()) >= 50,
    "يحتوي_أرقام": lambda ex: any(c.isdigit() for c in ex["input"]),
}


> ⚠️ **خطأ شائع جدًا بالذات للنماذج العربية**: تقييم النموذج فقط على العربية الفصحى (MSA) وإهمال اللهجات تمامًا، بينما الاستخدام الفعلي من المستخدمين العرب غالبًا ما يكون مزيجًا من الفصحى واللهجات المحكية. نموذج قد يحقق 90% على بيانات فصحى نظيفة وينهار لأداء ضعيف جدًا مع اللهجات دون أن يظهر ذلك في تقرير التقييم الإجمالي.

### 🔴 ج) طبقة التعمّق: القدرات الناشئة ومنحنيات القياس (Emergent Abilities & Scaling Curves)

ظاهرة **القدرات الناشئة** (Emergent Abilities): بعض القدرات (مثل الاستدلال متعدد الخطوات) تظهر **فجأة** عند تجاوز حجم نموذج معيّن، دون تحسّن تدريجي ملحوظ قبل تلك النقطة.


In [ ]:
def plot_scaling_curve_analysis(model_sizes: list, benchmark_scores: dict):
    """
    توضيح مفاهيمي: رسم منحنى الأداء مقابل حجم النموذج
    لاكتشاف نقاط "القفزة النوعية" (emergence thresholds)
    """
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, len(benchmark_scores), figsize=(15, 4))
    for idx, (benchmark_name, scores) in enumerate(benchmark_scores.items()):
        ax = axes[idx] if len(benchmark_scores) > 1 else axes
        ax.plot(model_sizes, scores, marker='o')
        ax.set_xscale('log')
        ax.set_xlabel('حجم النموذج (log scale)')
        ax.set_ylabel('الدقة')
        ax.set_title(benchmark_name)
        ax.axhline(y=0.25, color='gray', linestyle='--', label='خط الصدفة العشوائية')
    plt.tight_layout()
    return fig


> 💡 **تحذير بحثي مهم**: ورقة بحثية مؤثرة (Schaefer et al., 2023) أظهرت أن كثيرًا من "القدرات الناشئة" المُبلَّغ عنها هي في الواقع **قطع صناعي من طريقة القياس** (measurement artifact) — عند استخدام مقاييس مستمرة وسلسة (مثل log-likelihood) بدلاً من مقاييس ثنائية حادّة (صحيح/خاطئ تمامًا)، يظهر التحسّن **تدريجيًا وسلسًا** وليس كـ "قفزة مفاجئة". هذا درس بالغ الأهمية: **اختيار المقياس نفسه يُشكّل تفسيرك لسلوك النموذج**.


## 4. مثال محلول خطوة بخطوة

لنحلّل 10 حالات فشل من روبوت دعم عملاء عربي:

| # | نوع الخطأ | الشريحة | الشدة |
|---|---|---|---|
| 1 | هلوسة سياسة إرجاع | أسئلة سياسات | 5 |
| 2 | خطأ تنسيق JSON | تكامل API | 3 |
| 3 | رفض غير مبرر | استفسار شحن | 4 |
| 4 | خطأ لهجي (فهم خاطئ) | لهجة خليجية | 4 |
| 5 | إجابة صحيحة لكن طويلة جدًا | أسئلة بسيطة | 2 |
| … | … | … | … |

**الحساب:** 4/10 أخطاء في فئة "السياسات" → 40% فشل في شريحة تمثل 15% من الاستخدام. هذا أسوأ بكثير من المتوسط العام 12%. **القرار:** راجع قاعدة معرفة السياسات وأضف أمثلة تدريبية، قبل تحسين التنسيق.

**الدرس:** المتوسط يخفي التركيز. حلّل حسب الشريحة والشدة.


## 5. نصائح وحيل

> 💡 **اجعل تحليل الأخطاء جلسة أسبوعية**: 20-50 حالة حقيقية، لا عينات مختارة.

> 💡 **استخدم taxonomy ثابتة**: نفس التصنيف عبر الإصدارات يسمح بالمقارنة.

> 💡 **سجّل الشدة**: خطأ هلوسة في سياسة مالية أخطر من خطأ تنسيق.

> 💡 **احسب inter-annotator agreement**: إذا اختلف المُصنّفون كثيرًا، الـtaxonomy غامضة.

> 💡 **افحص الشرائح الصغيرة**: 20 مثالًا قد لا تكفي؛ اجمع المزيد قبل الحكم.


## 6. أخطاء شائعة ومصائد

> ⚠️ **الاكتفاء بالمتوسط العام**: 85% قد تخفي 40% فشل في شريحة حرجة.

> ⚠️ **تقييم العربية الفصحى فقط**: اللهجات والاختلاط اللغوي واقع المستخدمين. أضف شرائح لهجية.

> ⚠️ **اعتبار كل "قفزة" قدرة ناشئة**: قد تكون بسبب مقياس حاد (exact match). استخدم مقاييس مستمرة.

> ⚠️ **تصنيف الأخطاء دون شدة**: لا تعرف ما الأولوية.

> ⚠️ **عدم ربط التحليل بالعمل**: تقرير أخطاء بلا إصلاحات = هدر.


## 7. جدول مقارنة

| الأداة/المنهجية | الاستخدام |
|---|---|
| **Confusion Matrix موسّع** | لمهام التصنيف — يكشف أزواج الفئات الأكثر التباسًا |
| **Attention Visualization** | فهم أين "ينظر" النموذج عند الفشل (تحليلي أكثر من قطعي) |
| **Slice-based Evaluation** | كشف ضعف ممنهج في شرائح بيانات محدَّدة |
| **Contrastive Sets** | أزواج أمثلة متشابهة جدًا لكن بإجابات مختلفة — تختبر الفهم الحقيقي مقابل الحفظ السطحي |
| **Influence Functions** | تتبّع أي أمثلة تدريب أثّرت أكثر في قرار مُعيَّن (تحليلي متقدم) |


## 8. ملخص أفضل الممارسات

- اجعل تحليل الأخطاء اليدوي **عملية دورية مُجدولة**، وليس استجابة طارئة فقط عند حدوث مشكلة.
- صمّم شرائح تقييم خاصة بسياق استخدامك (لغة، لهجة، طول، تعقيد) دومًا.
- كن متشككًا تجاه الادعاءات حول "قدرات ناشئة" حتى تتحقق من سلوكها بمقاييس مستمرة متعددة.
- وثّق كل خطأ مكتشف بتصنيف منهجي (taxonomy) قابل لإعادة الاستخدام بين الإصدارات.

---


## 9. أسئلة مقابلات

**س1: لماذا قد تكون "القدرات الناشئة" artifact؟**

**الإجابة النموذجية:** عند استخدام مقياس حاد (صحيح/خطأ)، تحسّن سلس في احتمال كل خطوة يتحول إلى قفزة غير خطية في نجاح السلسلة الكاملة. عند استخدام مقياس مستمر (log-likelihood)، يظهر التحسن سلسًا. لذا بعض "القفزات" نتيجة القياس لا خاصية جوهرية.

**س2: كيف تصمم شرائح تقييم مفيدة؟**

**الإجابة النموذجية:** ابدأ بسياق الاستخدام: اللغة/اللهجة، نوع المهمة، طول السياق، مستوى الصعوبة، والمخاطر. اجعل كل شريحة كافية الحجم (100+ مثال إن أمكن)، وقِس الأداء والثقة لكل شريحة.

**س3: كيف تربط تحليل الأخطاء بالتحسين؟**

**الإجابة النموذجية:** لكل نمط فشل، حدّد السبب الجذري (بيانات ناقصة، prompt غامض، قدرة غير موجودة)، ثم إصلاحًا قابلًا للقياس (إضافة أمثلة، RAG، fine-tuning)، ثم أعد التقييم على نفس الشريحة للتأكد.


## 10. تمرين عملي

1. خذ 20 حالة فشل من تطبيق عربي (أو محاكاة) وصنّفها حسب taxonomy.
2. احسب نسبة كل نوع وشدة متوسطة.
3. قسّم النتائج حسب شريحة (لهجة/طول/مهمة) وابحث عن أسوأ شريحة.
4. اقترح إصلاحًا واحدًا لكل نمط وحدّد كيف ستقيس نجاحه.
5. أعد التقييم بعد الإصلاح وقارن.


## 11. مراجع للتعمّق

- Schaefer, R., et al. (2023). *Are Emergent Abilities of Large Language Models a Mirage?* NeurIPS.
- Srivastava, A., et al. (2023). *Beyond the Imitation Game* (BIG-Bench).
- Bender, E. M., et al. (2021). *On the Dangers of Stochastic Parrots*.
- Chung, H. W., et al. (2022). *Scaling Instruction-Finetuned Language Models* (slice analysis).


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التقييم الشامل / الفصل 45: التقييم النوعي وتحليل الأخطاء.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
